# Limpieza de los Datos truno 2

### Dado al plan de limpieza, se generara una limpieza de los datos que aplica las reglas ya aprobadas en el archivo [Limpieza de datos](docs\plan_limpieza.md) en esta parte no se eliminaran duplicados, solo se agruparan para decidir que hacer con ellos mas adelante

### Esta es una limpieza **Preliminar** que tendra que pasar la aprovacion de calidad requerida

## 1. Preparación reproducible

El módulo `src/limpieza.py` concentra las funciones de limpieza del TURNO 2. El notebook únicamente las ejecuta sobre el DataFrame crudo (`cargar_datos()`) y presenta el resultado: no hay ninguna transformación escrita directamente aquí.

Cada variable corregida (`DIRECCION`, `TELEFONO`, `SUPERVISOR`, `DIRECTOR`) conserva su valor crudo en una columna `_ORIGINAL`, y las reglas que solo marcan posibles duplicados o imputaciones agregan columnas nuevas en vez de sobrescribir sin rastro. Nada se elimina.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.diagnostico import cargar_datos
from src.limpieza import limpiar_datos_preliminar

df = cargar_datos()
limpio_pre = limpiar_datos_preliminar(df)

print(f"Registros: {len(limpio_pre):,}")
print(f"Columnas originales: {len(df.columns)} -> columnas tras limpieza preliminar: {len(limpio_pre.columns)}")


Registros: 11,890
Columnas originales: 19 -> columnas tras limpieza preliminar: 28


## 2. `ESTABLECIMIENTO`

Se conserva el nombre original y solo se agrega una clave normalizada para agrupar. Un grupo (misma clave + municipio) se marca como posible duplicado; queda confirmado únicamente cuando también coinciden JORNADA, PLAN y DIRECCION.

In [2]:
grupos = limpio_pre["ESTABLECIMIENTO_GRUPO_DUPLICADO"].dropna()
n_filas_grupo = len(grupos)
n_grupos = grupos.nunique()
n_confirmados = int(limpio_pre["ESTABLECIMIENTO_DUPLICADO_CONFIRMADO"].sum())

resumen_establecimiento = pd.DataFrame([
    {"indicador": "Filas en algún grupo (misma clave + municipio, >1 escritura)", "cantidad": n_filas_grupo},
    {"indicador": "Grupos distintos", "cantidad": n_grupos},
    {"indicador": "Filas confirmadas (también coincide JORNADA/PLAN/DIRECCION)", "cantidad": n_confirmados},
])
display(resumen_establecimiento)

ejemplo_id = grupos.value_counts().index[0]
ejemplo = limpio_pre.loc[
    limpio_pre["ESTABLECIMIENTO_GRUPO_DUPLICADO"] == ejemplo_id,
    ["CODIGO", "ESTABLECIMIENTO", "MUNICIPIO", "JORNADA", "PLAN", "ESTABLECIMIENTO_DUPLICADO_CONFIRMADO"],
]
display(Markdown(f"**Ejemplo — grupo #{ejemplo_id} (clave + municipio compartidos):**"))
display(ejemplo)

display(Markdown(
    f"**Conclusión:** {n_filas_grupo:,} filas ({n_filas_grupo / len(limpio_pre) * 100:.1f}%) comparten clave "
    f"normalizada y municipio con al menos otra escritura distinta, repartidas en {n_grupos:,} grupos. "
    "ESTABLECIMIENTO no se modifica (se conserva el nombre original); solo se agregan `ESTABLECIMIENTO_CLAVE` "
    "para comparar y `ESTABLECIMIENTO_GRUPO_DUPLICADO` para ubicar el grupo. De esos, únicamente "
    f"{n_confirmados:,} quedan `ESTABLECIMIENTO_DUPLICADO_CONFIRMADO=True` porque además coinciden JORNADA, "
    "PLAN y DIRECCION: los demás son, en su mayoría, el mismo nombre genérico (institutos públicos) repetido "
    "en jornadas o planteles distintos del mismo municipio, no duplicados reales. No se elimina ninguna fila; "
    "CODIGO sigue siendo la única llave primaria."
))


,indicador,cantidad
0,"Filas en algún grupo (misma clave + municipio,...",4045
1,Grupos distintos,1112
2,Filas confirmadas (también coincide JORNADA/PL...,42


**Ejemplo — grupo #764 (clave + municipio compartidos):**

,CODIGO,ESTABLECIMIENTO,MUNICIPIO,JORNADA,PLAN,ESTABLECIMIENTO_DUPLICADO_CONFIRMADO
5576,01-15-0062-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,MATUTINA,DIARIO(REGULAR),False
5577,01-15-0063-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,NOCTURNA,DIARIO(REGULAR),False
5578,01-15-0065-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,VESPERTINA,DIARIO(REGULAR),False
5579,01-15-0066-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,VESPERTINA,DIARIO(REGULAR),False
5580,01-15-0074-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,MATUTINA,DIARIO(REGULAR),False
5581,01-15-0075-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,VESPERTINA,DIARIO(REGULAR),False
5582,01-15-0076-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,VESPERTINA,DIARIO(REGULAR),False
5670,01-15-0376-46,INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFICADA,VILLA NUEVA,DOBLE,FIN DE SEMANA,False
5671,01-15-0378-46,INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFICADA,VILLA NUEVA,DOBLE,FIN DE SEMANA,False
5672,01-15-0382-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,VILLA NUEVA,DOBLE,FIN DE SEMANA,False


**Conclusión:** 4,045 filas (34.0%) comparten clave normalizada y municipio con al menos otra escritura distinta, repartidas en 1,112 grupos. ESTABLECIMIENTO no se modifica (se conserva el nombre original); solo se agregan `ESTABLECIMIENTO_CLAVE` para comparar y `ESTABLECIMIENTO_GRUPO_DUPLICADO` para ubicar el grupo. De esos, únicamente 42 quedan `ESTABLECIMIENTO_DUPLICADO_CONFIRMADO=True` porque además coinciden JORNADA, PLAN y DIRECCION: los demás son, en su mayoría, el mismo nombre genérico (institutos públicos) repetido en jornadas o planteles distintos del mismo municipio, no duplicados reales. No se elimina ninguna fila; CODIGO sigue siendo la única llave primaria.

## 3. `DIRECCION`

Se aplican en cascada las 4 reglas del plan: faltante disfrazado a NA, recorte del sufijo de municipio redundante, recorte de fecha incrustada al final, y corrección de la letra "O" por "0" en contexto numérico. El dominio rural no se penaliza por no tener número de casa.

In [3]:
faltante_na = int(limpio_pre["DIRECCION"].isna().sum())
faltante_crudo = int(df["DIRECCION"].fillna("").str.strip().eq("").sum())

cambio_texto = limpio_pre["DIRECCION_ORIGINAL"].fillna("") != limpio_pre["DIRECCION"].fillna("")
n_modificadas = int(cambio_texto.sum())

resumen_direccion = pd.DataFrame([
    {"regla": "Faltante disfrazado -> NA (vacía o igual al municipio)", "cantidad": faltante_na},
    {"regla": "Cualquier cambio de texto (municipio/fecha/O-por-cero)", "cantidad": n_modificadas},
])
display(resumen_direccion)

ejemplos = limpio_pre.loc[cambio_texto, ["DIRECCION_ORIGINAL", "DIRECCION"]].drop_duplicates().head(6)
display(Markdown("**Ejemplos antes / después:**"))
display(ejemplos)

display(Markdown(
    f"**Conclusión:** {faltante_na:,} direcciones quedaron `NA` por ser vacías o repetir exactamente el "
    f"municipio (el conteo crudo de vacías era {faltante_crudo:,}; la diferencia son las que solo decían el "
    f"nombre del municipio). Otras {n_modificadas:,} celdas cambiaron de texto: se recortó el sufijo de "
    "municipio redundante, la fecha incrustada al final, o se corrigió la letra “O” por “0” en contexto "
    "numérico. El valor original completo queda en `DIRECCION_ORIGINAL` para poder auditar cada cambio."
))


,regla,cantidad
0,Faltante disfrazado -> NA (vacía o igual al mu...,271
1,Cualquier cambio de texto (municipio/fecha/O-p...,855


**Ejemplos antes / después:**

,DIRECCION_ORIGINAL,DIRECCION
8,7A. CALLE 11-09 ZONA 6 COBAN,7A. CALLE 11-09 ZONA 6
59,"DIAGONAL 1 1-26, ZONA 1 COBÁN","DIAGONAL 1 1-26, ZONA 1"
65,2A. CALLE 1-99 ZONA 4 COBAN,2A. CALLE 1-99 ZONA 4
68,"2A. CALLE 8-18 ZONA 4, COBAN",2A. CALLE 8-18 ZONA 4
90,6A. CALLE 2-06 ZONA 8 PERIFERICO SUR COBAN,6A. CALLE 2-06 ZONA 8 PERIFERICO SUR
102,COBAN,<NA>


**Conclusión:** 271 direcciones quedaron `NA` por ser vacías o repetir exactamente el municipio (el conteo crudo de vacías era 99; la diferencia son las que solo decían el nombre del municipio). Otras 855 celdas cambiaron de texto: se recortó el sufijo de municipio redundante, la fecha incrustada al final, o se corrigió la letra “O” por “0” en contexto numérico. El valor original completo queda en `DIRECCION_ORIGINAL` para poder auditar cada cambio.

## 4. `TELEFONO`

Vacío queda como NA. El resto se procesa con `separar_numeros()` y se representa como una lista de contactos de 7-8 dígitos, no como un solo teléfono.

In [4]:
telefono_vacio_crudo = int(df["TELEFONO"].fillna("").str.strip().eq("").sum())
telefono_na = int(limpio_pre["TELEFONO"].isna().sum())
con_varios = int(limpio_pre["TELEFONO"].str.contains(";", na=False).sum())

resumen_telefono = pd.DataFrame([
    {"indicador": "Vacío en el original", "cantidad": telefono_vacio_crudo},
    {"indicador": "NA tras la limpieza (vacío o sin número recuperable)", "cantidad": telefono_na},
    {"indicador": "Celdas con más de un número recuperado", "cantidad": con_varios},
])
display(resumen_telefono)

ejemplos_tel = limpio_pre.loc[
    limpio_pre["TELEFONO_ORIGINAL"].fillna("").ne("")
    & (limpio_pre["TELEFONO_ORIGINAL"] != limpio_pre["TELEFONO"]),
    ["TELEFONO_ORIGINAL", "TELEFONO"],
].drop_duplicates().head(6)
display(Markdown("**Ejemplos antes / después:**"))
display(ejemplos_tel)

display(Markdown(
    f"**Conclusión:** {telefono_vacio_crudo:,} celdas venían vacías y quedan `NA` (faltante legítimo). El "
    "resto se procesó con `separar_numeros()` (reutilizada de `src/diagnostico.py`): se extraen los números "
    f"de 7 u 8 dígitos contenidos en la celda y se unen con `; `. {con_varios:,} celdas tenían más de un "
    f"contacto. Cuando no se recupera ningún número válido también queda `NA` ({telefono_na:,} en total). "
    "`TELEFONO_ORIGINAL` conserva el texto crudo tal como llegó."
))


,indicador,cantidad
0,Vacío en el original,969
1,NA tras la limpieza (vacío o sin número recupe...,993
2,Celdas con más de un número recuperado,189


**Ejemplos antes / después:**

,TELEFONO_ORIGINAL,TELEFONO
326,78208583-78209143,78208583; 78209143
484,79540830-79540909,79540830; 79540909
655,78391288-78392217,78391288; 78392217
674,79649696-78739432,79649696; 78739432
676,78739432-79649696,78739432; 79649696
772,78393245-78393246,78393245; 78393246


**Conclusión:** 969 celdas venían vacías y quedan `NA` (faltante legítimo). El resto se procesó con `separar_numeros()` (reutilizada de `src/diagnostico.py`): se extraen los números de 7 u 8 dígitos contenidos en la celda y se unen con `; `. 189 celdas tenían más de un contacto. Cuando no se recupera ningún número válido también queda `NA` (993 en total). `TELEFONO_ORIGINAL` conserva el texto crudo tal como llegó.

## 5. `SUPERVISOR`

Se corrigen grafías puntuales, se fusionan variantes de tildes/espacios dentro del mismo DISTRITO, y la ausencia real se imputa con el supervisor más frecuente de su distrito cuando existe esa referencia.

In [5]:
sup_ausente_final = int(limpio_pre["SUPERVISOR"].isna().sum())
sup_imputado = int(limpio_pre["SUPERVISOR_IMPUTADO"].sum())

resumen_supervisor = pd.DataFrame([
    {"indicador": "Sin supervisor identificable (tras corregir y fusionar)", "cantidad": sup_ausente_final + sup_imputado},
    {"indicador": "Imputados con el supervisor más frecuente del DISTRITO", "cantidad": sup_imputado},
    {"indicador": "Quedan NA (sin ninguna referencia en su DISTRITO)", "cantidad": sup_ausente_final},
])
display(resumen_supervisor)

ejemplos_sup = limpio_pre.loc[
    limpio_pre["SUPERVISOR_ORIGINAL"].fillna("").ne("")
    & limpio_pre["SUPERVISOR"].notna()
    & (limpio_pre["SUPERVISOR_ORIGINAL"] != limpio_pre["SUPERVISOR"])
    & ~limpio_pre["SUPERVISOR_IMPUTADO"],
    ["SUPERVISOR_ORIGINAL", "SUPERVISOR"],
].drop_duplicates().head(6)
display(Markdown("**Ejemplos de corrección de grafía / fusión de variantes:**"))
display(ejemplos_sup)

display(Markdown(
    "**Conclusión:** tras corregir grafías puntuales (O/0, tilde grave, apóstrofo, puntuación final) y "
    "fusionar variantes de tildes/espacios dentro del mismo DISTRITO, quedan "
    f"{sup_ausente_final + sup_imputado:,} filas sin un supervisor identificable en el texto original. De "
    f"esas, solo {sup_imputado:,} se pudieron imputar con el supervisor más frecuente de su DISTRITO (un "
    f"distrito, un supervisor): la gran mayoría de las filas sin supervisor tampoco tienen DISTRITO "
    f"capturado, así que no existe ninguna referencia con la cual imputar y quedan `NA` "
    f"({sup_ausente_final:,}). Esta limitación depende de que DISTRITO (TURNO 1) se complete; no es un error "
    "de esta regla."
))


,indicador,cantidad
0,Sin supervisor identificable (tras corregir y ...,561
1,Imputados con el supervisor más frecuente del ...,0
2,Quedan NA (sin ninguna referencia en su DISTRITO),561


**Ejemplos de corrección de grafía / fusión de variantes:**

,SUPERVISOR_ORIGINAL,SUPERVISOR
930,"AMALIA ESTER LIX SOCOP DE CHUY,",AMALIA ESTER LIX SOCOP DE CHUY
1225,EDNA ODILIA ACEVED0,EDNA ODILIA ACEVEDO
3517,EDGAR ROLANDO JUÁREZ ORTÌZ,EDGAR ROLANDO JUÁREZ ORTÍZ
3970,ENMA ELIZABETH GONZÁLEZ FAJARDO DE QUIÑONEZ.,ENMA ELIZABETH GONZÁLEZ FAJARDO DE QUIÑONEZ
4054,MABELLINE DAHAN ALDANA RODAS DE PINZON.,MABELLINE DAHAN ALDANA RODAS DE PINZON
6013,BRYAN O´NELL GAITAN RODRIGUEZ,BRYAN O'NELL GAITAN RODRIGUEZ


**Conclusión:** tras corregir grafías puntuales (O/0, tilde grave, apóstrofo, puntuación final) y fusionar variantes de tildes/espacios dentro del mismo DISTRITO, quedan 561 filas sin un supervisor identificable en el texto original. De esas, solo 0 se pudieron imputar con el supervisor más frecuente de su DISTRITO (un distrito, un supervisor): la gran mayoría de las filas sin supervisor tampoco tienen DISTRITO capturado, así que no existe ninguna referencia con la cual imputar y quedan `NA` (561). Esta limitación depende de que DISTRITO (TURNO 1) se complete; no es un error de esta regla.

## 6. `DIRECTOR`

Se separa el título incrustado, se reclasifica a NA la ausencia real (incluye los disfrazados) y se fusionan variantes dentro del mismo MUNICIPIO.

In [6]:
dir_titulo = int(limpio_pre["DIRECTOR_TITULO"].notna().sum())
dir_ausente_final = int(limpio_pre["DIRECTOR"].isna().sum())
dir_ausente_crudo = int(df["DIRECTOR"].fillna("").str.strip().eq("").sum())

resumen_director = pd.DataFrame([
    {"indicador": "Títulos separados (LIC./LICDA./PEM.)", "cantidad": dir_titulo},
    {"indicador": "Ausencia real -> NA (vacíos + disfrazados)", "cantidad": dir_ausente_final},
])
display(resumen_director)

ejemplos_titulo = limpio_pre.loc[
    limpio_pre["DIRECTOR_TITULO"].notna(), ["DIRECTOR_ORIGINAL", "DIRECTOR", "DIRECTOR_TITULO"]
].drop_duplicates().head(5)
display(Markdown("**Ejemplos de título separado:**"))
display(ejemplos_titulo)

ejemplos_disfrazado = limpio_pre.loc[
    limpio_pre["DIRECTOR"].isna() & limpio_pre["DIRECTOR_ORIGINAL"].fillna("").str.strip().ne(""),
    "DIRECTOR_ORIGINAL",
].drop_duplicates().head(6)
display(Markdown("**Ejemplos de faltante disfrazado reclasificado a NA:**"))
display(ejemplos_disfrazado)

display(Markdown(
    f"**Conclusión:** se separaron {dir_titulo:,} títulos incrustados a `DIRECTOR_TITULO`, dejando el nombre "
    f"limpio en DIRECTOR. {dir_ausente_final:,} filas ({dir_ausente_final / len(limpio_pre) * 100:.1f}%) "
    f"quedan `NA` (el conteo crudo de vacías era {dir_ausente_crudo:,}: el resto son disfrazados como "
    "guiones, ceros o \"SIN DATOS\", que la regla de \"menos de 2 palabras con letra\" también detecta). No "
    "es un dato imputable: es propio del establecimiento. Las variantes de tildes/espacios se fusionan "
    "dentro del mismo MUNICIPIO -no DISTRITO como en SUPERVISOR- para no mezclar homónimos de otra "
    "jurisdicción."
))


,indicador,cantidad
0,Títulos separados (LIC./LICDA./PEM.),20
1,Ausencia real -> NA (vacíos + disfrazados),2174


**Ejemplos de título separado:**

,DIRECTOR_ORIGINAL,DIRECTOR,DIRECTOR_TITULO
513,LIC. ELGI WALTER BOTEO GARCÍA,ELGI WALTER BOTEO GARCÍA,LIC.
567,PEM. ZOILA CÁNDIDA LUNA PÉREZ,ZOILA CÁNDIDA LUNA PÉREZ,PEM.
711,LIC. CARLOS HUMBERTO CABRERA OVALLE,CARLOS HUMBERTO CABRERA OVALLE,LIC.
2404,LICDA. DORIS ZUNUN CARRERA,DORIS ZUNUN CARRERA,LICDA.
2742,LICDA. LILIA ADRIANA LEMUS GALÁN,LILIA ADRIANA LEMUS GALÁN,LICDA.


**Ejemplos de faltante disfrazado reclasificado a NA:**

0                 --
102              ---
338                -
790                .
1326            ----
1332    ------------
Name: DIRECTOR_ORIGINAL, dtype: string

**Conclusión:** se separaron 20 títulos incrustados a `DIRECTOR_TITULO`, dejando el nombre limpio en DIRECTOR. 2,174 filas (18.3%) quedan `NA` (el conteo crudo de vacías era 1,755: el resto son disfrazados como guiones, ceros o "SIN DATOS", que la regla de "menos de 2 palabras con letra" también detecta). No es un dato imputable: es propio del establecimiento. Las variantes de tildes/espacios se fusionan dentro del mismo MUNICIPIO -no DISTRITO como en SUPERVISOR- para no mezclar homónimos de otra jurisdicción.

## 7. Resumen consolidado de la limpieza TURNO 2

La tabla siguiente reúne las cantidades afectadas por cada regla, recalculadas en cada ejecución para que el número nunca quede desactualizado respecto al código.

In [7]:
resumen_turno2 = pd.DataFrame([
    {"variable": "ESTABLECIMIENTO", "regla": "Marcar posibles duplicados (clave + municipio)", "cantidad_afectada": n_filas_grupo},
    {"variable": "ESTABLECIMIENTO", "regla": "Duplicados confirmados (JORNADA+PLAN+DIRECCION)", "cantidad_afectada": n_confirmados},
    {"variable": "DIRECCION", "regla": "Faltante disfrazado -> NA", "cantidad_afectada": faltante_na},
    {"variable": "DIRECCION", "regla": "Texto corregido (municipio/fecha/O-por-cero)", "cantidad_afectada": n_modificadas},
    {"variable": "TELEFONO", "regla": "Vacío -> NA", "cantidad_afectada": telefono_vacio_crudo},
    {"variable": "TELEFONO", "regla": "Formato no estándar -> lista de números", "cantidad_afectada": con_varios},
    {"variable": "SUPERVISOR", "regla": "Ausencia real (tras corrección/fusión)", "cantidad_afectada": sup_ausente_final + sup_imputado},
    {"variable": "SUPERVISOR", "regla": "Imputado por DISTRITO", "cantidad_afectada": sup_imputado},
    {"variable": "DIRECTOR", "regla": "Título separado", "cantidad_afectada": dir_titulo},
    {"variable": "DIRECTOR", "regla": "Ausencia real -> NA", "cantidad_afectada": dir_ausente_final},
])
display(resumen_turno2)


,variable,regla,cantidad_afectada
0,ESTABLECIMIENTO,Marcar posibles duplicados (clave + municipio),4045
1,ESTABLECIMIENTO,Duplicados confirmados (JORNADA+PLAN+DIRECCION),42
2,DIRECCION,Faltante disfrazado -> NA,271
3,DIRECCION,Texto corregido (municipio/fecha/O-por-cero),855
4,TELEFONO,Vacío -> NA,969
5,TELEFONO,Formato no estándar -> lista de números,189
6,SUPERVISOR,Ausencia real (tras corrección/fusión),561
7,SUPERVISOR,Imputado por DISTRITO,0
8,DIRECTOR,Título separado,20
9,DIRECTOR,Ausencia real -> NA,2174
